In [1]:
import pandas as pd
import os

# Folder and file setup
data_folder = 'data/'
input_file = 'data-2.xlsx'

# Read the merged Excel file into DataFrame (load all columns as is)
file_path = os.path.join(data_folder, input_file)
df = pd.read_excel(file_path, engine='openpyxl')

print(f"DataFrame shape: {df.shape}")
df.head()


DataFrame shape: (150003, 27)


,BudgetCode,ProjectName,costCenter,DecisionMoment,Period,LastUpdate,OriginCostCode,whatLVL1Desc,whatCode,whatDescriptionEng,...,TechRef,AuthReq,Source,Actuals/forecast,Description,Description2,AccountCode,PaymentPlaceName,CurrencyCode,Total CHF
0,AM120,HEP-C YEREVAN-ARMAVIR,Project,2023-01,POA,NaN,FIN,BASIC SUPPORT COSTS,BSREN,BASIC SUPPORT COSTS BUILDING RENT,...,MisPharMan,MisPharMan,FIN,Forecast,Medical Warehouse,Outsourcing,65000.0,Yerevan,AMD,13170.60
1,AM120,HEP-C YEREVAN-ARMAVIR,Project,2023-02,NaN,NaN,FIN,BASIC SUPPORT COSTS,BSREN,BASIC SUPPORT COSTS BUILDING RENT,...,MisPharMan,MisPharMan,FIN,Forecast,Medical Warehouse,Outsourcing,65000.0,YVN,AMD,12073.05
2,AM120,HEP-C YEREVAN-ARMAVIR,Project,2023-03,NaN,NaN,FIN,BASIC SUPPORT COSTS,BSREN,BASIC SUPPORT COSTS BUILDING RENT,...,MisPharMan,MisPharMan,FIN,Forecast,Medical Warehouse,Outsourcing,65000.0,YVN,AMD,9877.95
3,AM120,HEP-C YEREVAN-ARMAVIR,Project,2023-04,NaN,Last Update PRE-MYR,FIN,BASIC SUPPORT COSTS,BSREN,BASIC SUPPORT COSTS BUILDING RENT,...,MisPharMan,MisPharMan,FIN,Forecast,Medical Warehouse,Outsourcing,65000.0,YVN,AMD,9877.95
4,AM120,HEP-C YEREVAN-ARMAVIR,Project,2023-04,NaN,Last Update PRE-MYR,FIN,BASIC SUPPORT COSTS,BSREN,BASIC SUPPORT COSTS BUILDING RENT,...,MisPharMan,MisPharMan,UNIF,Actuals,RENT Warehouse Pharma 42/2 3 St Abgar King str...,CT AM101/2023/06,65000.0,AM1CO,AMD,247.52


In [2]:
# Filter: BudgetCode ends with '9' and Actuals/forecast == 'Actuals'
filtered_df = df[
    df['BudgetCode'].astype(str).str.endswith('9') &
    (df['Actuals/forecast'] == 'Actuals')
]

print(f"Filtered DataFrame shape: {filtered_df.shape}")
filtered_df.head()

Filtered DataFrame shape: (25414, 27)


,BudgetCode,ProjectName,costCenter,DecisionMoment,Period,LastUpdate,OriginCostCode,whatLVL1Desc,whatCode,whatDescriptionEng,...,TechRef,AuthReq,Source,Actuals/forecast,Description,Description2,AccountCode,PaymentPlaceName,CurrencyCode,Total CHF
10404,BF119,DJIBO,Project,2023-02,NaN,NaN,FIN,BASIC SUPPORT COSTS,BSREN,BASIC SUPPORT COSTS BUILDING RENT,...,ADMN,FIN-ASSIST,UNIF,Actuals,LOYER DE JANV-MARS 2023 DJIBO,BF1DJ-PUR-230040,65000.0,BF1DJ,XOF,1944.88
10405,BF119,DJIBO,Project,2023-02,NaN,NaN,FIN,BASIC SUPPORT COSTS,BSREN,BASIC SUPPORT COSTS BUILDING RENT,...,ADMN,FIN-ASSIST,UNIF,Actuals,LOYER JANV-MARS23 DJIBO,NaN,65000.0,BF1DJ,XOF,1732.08
10406,BF119,DJIBO,Project,2023-02,NaN,NaN,HRNAT,HR WORKFORCE,HLLWW,WATCHMAN,...,LOG,NaN,HRNAT,Actuals,1-GARDIEN,KIENI Issa,66005.0,66005,XOF,0.00
10407,BF119,DJIBO,Project,2023-02,NaN,NaN,HRNAT,HR WORKFORCE,HLLWW,WATCHMAN,...,LOG,NaN,HRNAT,Actuals,1-GARDIEN,TAMBOURA Adama 2,66005.0,66005,XOF,0.00
10408,BF119,DJIBO,Project,2023-02,NaN,NaN,HRNAT,HR WORKFORCE,HLLWW,WATCHMAN,...,LOG,NaN,HRNAT,Actuals,1-GARDIEN,TAMBOURA Ousmane Moussa,66005.0,66005,XOF,0.00


In [9]:
# Add a 'Year' column extracted from DecisionMoment
filtered_df['Year'] = filtered_df['DecisionMoment'].astype(str).str[:4]

# Group by BudgetCode, Year, and whatDescriptionEng, then sum Total CHF
grouped_df = filtered_df.groupby(['BudgetCode', 'Year', 'whatLVL1Desc'], as_index=False)['Total CHF'].sum()

print(grouped_df.head())

# Save the grouped DataFrame to an Excel file in the data folder
output_path = os.path.join(data_folder, "v1_calculations.xlsx")
grouped_df.to_excel(output_path, index=False)
print(f"Grouped results saved to {output_path}")

  BudgetCode  Year         whatLVL1Desc  Total CHF
0      BF119  2023  BASIC SUPPORT COSTS   58644.00
1      BF119  2023         CONSTRUCTION    9495.26
2      BF119  2023            EQUIPMENT   79761.32
3      BF119  2023         HR WORKFORCE  206642.73
4      BF119  2024  BASIC SUPPORT COSTS   61620.88
Grouped results saved to data/v1_calculations.xlsx


C:\Users\TML11\AppData\Local\Temp\ipykernel_25512\2440007649.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_df['Year'] = filtered_df['DecisionMoment'].astype(str).str[:4]


In [ ]:
# Group by BudgetCode, Year, and whatLVL1Desc, then sum Total CHF
grouped_df = filtered_df.groupby(['BudgetCode', 'Year', 'whatLVL1Desc'], as_index=False)['Total CHF'].sum()

# Create a list to store the new rows with subtotals
rows_with_totals = []

# Iterate over each BudgetCode and Year group
for (budget_code, year), group in grouped_df.groupby(['BudgetCode', 'Year']):
    # Add all cost area rows for this BudgetCode-Year
    rows_with_totals.extend(group.to_dict('records'))
    # Calculate subtotal for this BudgetCode-Year
    subtotal = group['Total CHF'].sum()
    # Create subtotal row
    subtotal_row = {
        'BudgetCode': budget_code,
        'Year': year,
        'whatLVL1Desc': 'TOTAL',
        'Total CHF': subtotal
    }
    rows_with_totals.append(subtotal_row)

# Create a new DataFrame with the subtotals included
final_df = pd.DataFrame(rows_with_totals)

print(final_df.head(10))

# Save the final DataFrame to an Excel file
output_path = os.path.join(data_folder, "v2_calculations.xlsx")
final_df.to_excel(output_path, index=False)
print(f"Grouped results with totals saved to {output_path}")

  BudgetCode  Year         whatLVL1Desc  Total CHF
0      BF119  2023  BASIC SUPPORT COSTS   58644.00
1      BF119  2023         CONSTRUCTION    9495.26
2      BF119  2023            EQUIPMENT   79761.32
3      BF119  2023         HR WORKFORCE  206642.73
4      BF119  2023                TOTAL  354543.31
5      BF119  2024  BASIC SUPPORT COSTS   61620.88
6      BF119  2024         HR WORKFORCE  441196.99
7      BF119  2024                TOTAL  502817.87
8      CD149  2021  BASIC SUPPORT COSTS  141038.95
9      CD149  2021                TOTAL  141038.95
Grouped results with totals saved to data/v2_calculations.xlsx


In [ ]:
# Group by BudgetCode, Year, whatLVL1Desc, and whatDescriptionEng, then sum Total CHF
grouped_df = filtered_df.groupby(
    ['BudgetCode', 'Year', 'whatLVL1Desc', 'whatDescriptionEng'],
    as_index=False
)['Total CHF'].sum()

# Create a list to store the new rows with subtotals
rows_with_totals = []

# Iterate over each BudgetCode, Year, and whatLVL1Desc group
for (budget_code, year, lvl1desc), group in grouped_df.groupby(['BudgetCode', 'Year', 'whatLVL1Desc']):
    # Add all cost area rows for this BudgetCode-Year-whatLVL1Desc
    rows_with_totals.extend(group.to_dict('records'))
    # Calculate subtotal for this BudgetCode-Year-whatLVL1Desc
    subtotal = group['Total CHF'].sum()
    # Create subtotal row
    subtotal_row = {
        'BudgetCode': budget_code,
        'Year': year,
        'whatLVL1Desc': lvl1desc,
        'whatDescriptionEng': 'TOTAL',
        'Total CHF': subtotal
    }
    rows_with_totals.append(subtotal_row)

# Create a new DataFrame with the subtotals included
final_df = pd.DataFrame(rows_with_totals)

print(final_df.head(10))

# Save the final DataFrame to an Excel file
output_path = os.path.join(data_folder, "v3_calculations.xlsx")
final_df.to_excel(output_path, index=False)
print(f"Grouped results with totals saved to {output_path}")

  BudgetCode  Year         whatLVL1Desc  Total CHF whatDescriptionEng
0      BF119  2023  BASIC SUPPORT COSTS   58644.00                NaN
1      BF119  2023  BASIC SUPPORT COSTS   58644.00              TOTAL
2      BF119  2023         CONSTRUCTION    9495.26                NaN
3      BF119  2023         CONSTRUCTION    9495.26              TOTAL
4      BF119  2023            EQUIPMENT   79761.32                NaN
5      BF119  2023            EQUIPMENT   79761.32              TOTAL
6      BF119  2023         HR WORKFORCE  206642.73                NaN
7      BF119  2023         HR WORKFORCE  206642.73              TOTAL
8      BF119  2024  BASIC SUPPORT COSTS   61620.88                NaN
9      BF119  2024  BASIC SUPPORT COSTS   61620.88              TOTAL
Grouped results with totals saved to data/v4_calculations.xlsx


In [14]:
# Group by BudgetCode, Year, whatLVL1Desc, and whatDescriptionEng, then sum Total CHF
grouped_df = filtered_df.groupby(
    ['BudgetCode', 'Year', 'whatLVL1Desc', 'whatDescriptionEng'],
    as_index=False
)['Total CHF'].sum()

# Create a list to store the new rows with subtotals
rows_with_totals = []

# Iterate over each BudgetCode and Year group
for (budget_code, year), group in grouped_df.groupby(['BudgetCode', 'Year']):
    # Add all cost area rows for this BudgetCode-Year
    rows_with_totals.extend(group.to_dict('records'))
    # Calculate grand total for this BudgetCode-Year (sum over all whatDescriptionEng)
    grand_total = group['Total CHF'].sum()
    # Create grand total row
    grand_total_row = {
        'BudgetCode': budget_code,
        'Year': year,
        'whatLVL1Desc': 'TOTAL',
        'whatDescriptionEng': 'TOTAL',
        'Total CHF': grand_total
    }
    rows_with_totals.append(grand_total_row)

# Create a new DataFrame with the grand totals included
final_df = pd.DataFrame(rows_with_totals)

print(final_df.head(10))

# Save the final DataFrame to an Excel file
output_path = os.path.join(data_folder, "v4_calculations.xlsx")
final_df.to_excel(output_path, index=False)
print(f"Grouped results with grand totals saved to {output_path}")

  BudgetCode  Year         whatLVL1Desc  \
0      BF119  2023  BASIC SUPPORT COSTS   
1      BF119  2023         CONSTRUCTION   
2      BF119  2023            EQUIPMENT   
3      BF119  2023            EQUIPMENT   
4      BF119  2023         HR WORKFORCE   
5      BF119  2023         HR WORKFORCE   
6      BF119  2023         HR WORKFORCE   
7      BF119  2023         HR WORKFORCE   
8      BF119  2023         HR WORKFORCE   
9      BF119  2023         HR WORKFORCE   

                          whatDescriptionEng  Total CHF  
0          BASIC SUPPORT COSTS BUILDING RENT   58644.00  
1         TEMPORARY INSTALLATION - STRUCTURE    9495.26  
2    PURCHASE APPLIANCES & RELATED EQUIPMENT    5877.12  
3  PURCHASE ELECTRICITY PRODUCTION EQUIPMENT   73884.20  
4                        MEDICINES DISPENSER       0.00  
5                       PHARMACY STOREKEEPER   43508.90  
6                        PHARMACY SUPERVISOR   76868.13  
7                   PROJECT PHARMACY MANAGER       0.00  
8   

In [ ]:
# Folder and file setup
data_folder = 'data/'
input_file = 'mergedportail.xlsx'

# Read the merged Excel file into DataFrame (load all columns as is)
file_path = os.path.join(data_folder, input_file)
df_inventory = pd.read_excel(file_path, engine='openpyxl')

print(f"DataFrame shape: {df_inventory.shape}")
df_inventory.head()

# Save the final DataFrame to an Excel file
output_path = os.path.join(data_folder, "v4_calculations.xlsx")
final_df.to_excel(output_path, index=False)
print(f"Grouped results with grand totals saved to {output_path}")


DataFrame shape: (262636, 54)


,order_oc_id,customer_id,project_id,country_of_delivery,supply_center_id,order_type,direct_delivery,warehouse_id,country_of_origin,order_priority_type,...,order_description,order_completion,Total_LeadTime,order_weight_kg,order_volume_dm3,order_volume_m3,price_orderline,unique_order_code,unique_shipment_code,unique_backorder_code
0,18/5035/CH/NG111,CH001MCH,NG111MCH,NG,MSFL,med,No,BDX,FR,Emergency,...,EMERG MED OR NGALA (thru CM),Complete,81.0,0.29,2.80,0.00280,477.56,"NG111MCH,2018-11-23","NG,2019-01-04","18/5035/CH/NG111,SINSNEIOKN12,2019-01-04"
1,18/5035/CH/NG111,CH001MCH,NG111MCH,NG,MSFL,med,No,BDX,FR,Emergency,...,EMERG MED OR NGALA (thru CM),Complete,69.0,6.40,52.00,0.05200,377.60,"NG111MCH,2018-11-23","NG,2018-12-07","18/5035/CH/NG111,SINSIVCST18W1,2018-12-07"
2,18/5035/CH/NG111,CH001MCH,NG111MCH,NG,MSFL,med,No,BDX,FR,Emergency,...,EMERG MED OR NGALA (thru CM),Complete,69.0,0.40,3.25,0.00325,23.60,"NG111MCH,2018-11-23","NG,2018-12-07","18/5035/CH/NG111,SINSIVCST18W1,2018-12-07"
3,18/5035/CH/NG111,CH001MCH,NG111MCH,NG,MSFL,med,No,BDX,FR,Emergency,...,EMERG MED OR NGALA (thru CM),Complete,557.0,0.30,2.00,0.00200,15.70,"NG111MCH,2018-11-23","NG,2019-12-13","18/5035/CH/NG111,DINJTRAM1A-,2019-12-13"
4,18/5035/CH/NG111,CH001MCH,NG111MCH,NG,MSFL,med,No,BDX,FR,Emergency,...,EMERG MED OR NGALA (thru CM),Complete,557.0,0.60,4.00,0.00400,31.40,"NG111MCH,2018-11-23","NG,2019-12-13","18/5035/CH/NG111,DINJTRAM1A-,2019-12-13"


In [16]:
df_inventory = df

In [22]:
# Remove the suffix "MCH" from all project_id values
df_inventory['project_id'] = df_inventory['project_id'].str.replace('MCH', '', regex=False)

# Filter: project_id ends with '9' and order_completion == 'Complete'
filtered_inventory = df_inventory[
    df_inventory['project_id'].astype(str).str.endswith('9') &
    (df_inventory['order_completion'] == 'Complete')
]

print(f"Filtered DataFrame shape: {filtered_inventory.shape}")
filtered_inventory.head()

Filtered DataFrame shape: (33929, 54)


,order_oc_id,customer_id,project_id,country_of_delivery,supply_center_id,order_type,direct_delivery,warehouse_id,country_of_origin,order_priority_type,...,order_description,order_completion,Total_LeadTime,order_weight_kg,order_volume_dm3,order_volume_m3,price_orderline,unique_order_code,unique_shipment_code,unique_backorder_code
1734,18/9008/CH/CD109,CH001MCH,CD109,CD,MSFL,med,No,BDX,FR,High,...,Kits FH,Complete,56.0,3.0,48.0,0.048,306.54,"CD109MCH,2018-12-18","CD,2019-02-01","18/9008/CH/CD109,SINSBIOP1S-,2019-02-01"
1735,18/9008/CH/CD109,CH001MCH,CD109,CD,MSFL,med,No,BDX,FR,High,...,Kits FH,Complete,94.0,21.6,45.0,0.045,85.50,"CD109MCH,2018-12-18","CD,2019-03-01","18/9008/CH/CD109,CWATDISING1,2019-03-01"
1736,18/9008/CH/CD109,CH001MCH,CD109,CD,MSFL,med,No,BDX,FR,High,...,Kits FH,Complete,78.0,519.0,5560.0,5.560,8424.59,"CD109MCH,2018-12-18","CD,2019-02-04","18/9008/CH/CD109,KMEDMEBO17A,2019-02-04"
1737,18/9008/CH/CD109,CH001MCH,CD109,CD,MSFL,med,No,BDX,FR,High,...,Kits FH,Complete,56.0,3.0,48.0,0.048,306.54,"CD109MCH,2018-12-18","CD,2019-02-01","18/9008/CH/CD109,SINSBIOP1S-,2019-02-01"
1738,18/CH/UG203/PO04292,CH001MCH,UG209,UG,MSFL,log,No,BDX,FR,High,...,REPLENISHMENT TO CHARGE CD472,Complete,107.0,377.0,2322.0,2.322,2967.26,"UG209MCH,2018-12-18","UG,2019-03-29","18/CH/UG203/PO04292,KADMMLIFC08,2019-03-29"


In [ ]:
# Group by project_id and sum order_volume_m3 and price_orderline
inventory_grouped_df = filtered_inventory.groupby('project_id', as_index=False)[['order_volume_m3', 'price_orderline']].sum()

print(inventory_grouped_df.head())

  project_id  order_volume_m3  price_orderline
0      AM109         7.131550       103481.260
1      BF109        24.452809        58359.426
2      BF119        69.404241       294676.249
3      CD149         0.166040         6197.090
4      CD509       205.361972       522065.477


In [23]:
# Keep only rows where actual_delivery_date starts with 2023 or 2024
filtered_inventory = filtered_inventory[
    filtered_inventory['actual_delivery_date'].astype(str).str.startswith(('2023', '2024'))
]

# Optional: Add a 'Year' column for grouping
filtered_inventory['Year'] = filtered_inventory['actual_delivery_date'].astype(str).str[:4]

# Group by Year and project_id, then sum order_volume_m3 and price_orderline
inventory_grouped_df = filtered_inventory.groupby(['Year', 'project_id'], as_index=False)[['order_volume_m3', 'price_orderline']].sum()

print(inventory_grouped_df.head())

   Year project_id  order_volume_m3  price_orderline
0  2023      AM109         7.131550       103481.260
1  2023      BF109        14.812492        27635.343
2  2023      BF119        41.986260       186294.383
3  2023      CD149         0.166040         6197.090
4  2023      CD509        44.022758       171806.400


In [ ]:
# Save the final DataFrame to an Excel file
output_path = os.path.join(data_folder, "almas_fil.xlsx")
inventory_grouped_df.to_excel(output_path, index=False)
print(f"Grouped results with grand totals saved to {output_path}")

Grouped results with grand totals saved to data/almas_fil.xlsx
